# Mu-SHROOM (English): XLM-RoBERTa-Large token classification

Fine-tune **`xlm-roberta-large`** on **`Helsinki-NLP/mu-shroom`** config **`en`** to label **hallucinated vs non-hallucinated** spans using **`hard_labels`** (character spans → BIO tags).

**Before running:** Runtime → **Change runtime type** → **GPU** (T4 is enough).

Outputs are saved under **`/content/`** on the Colab VM (see the cell *Where are my saved files?* after training).

**Fixes baked in:** class‑weighted loss (stops predicting only `O`), token‑level F1 for B/I‑HALL + `f1_hall_mean` for checkpoint selection, seqeval `zero_division=0`, more epochs on tiny data, GPU device fix in inference, `scikit-learn` for metrics. Optional: set `HF_TOKEN` in Colab secrets for faster Hub downloads (not required for public assets).

In [1]:
# Install dependencies (Colab) — pin stack so save/load LayerNorm names match train vs inference
!pip install -q "datasets>=2.18.0" "transformers>=4.41.0" "accelerate>=0.29.0" evaluate seqeval scikit-learn

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
torch: 2.10.0+cu128
cuda available: True
GPU: Tesla T4


In [2]:
import inspect
import os

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

OUTPUT_DIR = "/content/xlmr-mu-shroom-en"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "xlm-roberta-large"
MAX_LENGTH = 256
SEED = 42
NUM_EPOCHS = 8  # small data — a few more epochs help minority classes

label_list = ["O", "B-HALL", "I-HALL"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

seqeval = evaluate.load("seqeval")


def spans_to_token_bio(offsets, spans):
    spans = spans or []
    spans = sorted(spans, key=lambda x: (x[0], x[1]))

    def token_overlaps_span(tok_s, tok_e, sp_s, sp_e):
        return (tok_s < sp_e) and (tok_e > sp_s)

    labels = []
    for tok_s, tok_e in offsets:
        if tok_s == tok_e == 0:
            labels.append(-100)
            continue
        matched = None
        for sp_s, sp_e in spans:
            if token_overlaps_span(tok_s, tok_e, sp_s, sp_e):
                matched = (sp_s, sp_e)
                break
        if matched is None:
            labels.append(label2id["O"])
            continue
        sp_s, sp_e = matched
        begins = tok_s <= sp_s < tok_e or tok_s == sp_s
        labels.append(label2id["B-HALL"] if begins else label2id["I-HALL"])

    prev_inside = False
    for i, lab in enumerate(labels):
        if lab in (-100, label2id["O"]):
            prev_inside = False
            continue
        if lab == label2id["I-HALL"] and not prev_inside:
            labels[i] = label2id["B-HALL"]
        prev_inside = True
    return labels


def load_english_mu_shroom():
    try:
        return load_dataset("Helsinki-NLP/mu-shroom", "en")
    except Exception:
        ds = load_dataset("Helsinki-NLP/mu-shroom", "all")
        for c in ["language", "lang", "locale"]:
            if "validation" in ds and c in ds["validation"].column_names:
                return ds.filter(lambda x: x[c] == "en")
        raise RuntimeError("Could not filter English from 'all' config.")


def make_training_args(output_dir, train_bs, eval_bs, epochs, lr, wd, seed, use_fp16):
    """Best checkpoint tracks token-level F1 on minority (Hall) classes, not plain accuracy."""
    sig = inspect.signature(TrainingArguments.__init__)
    p = sig.parameters
    kw = dict(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=wd,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="f1_hall_mean",
        greater_is_better=True,
        fp16=use_fp16,
        seed=seed,
        report_to="none",
    )
    if "evaluation_strategy" in p:
        kw["evaluation_strategy"] = "epoch"
        kw["save_strategy"] = "epoch"
    else:
        if "eval_strategy" in p:
            kw["eval_strategy"] = "epoch"
        if "save_strategy" in p:
            kw["save_strategy"] = "epoch"
    return TrainingArguments(**kw)


def collect_flat_labels(dataset, label_key="labels"):
    y = []
    for row in dataset[label_key]:
        y.extend(int(t) for t in row if t != -100)
    return np.array(y, dtype=np.int64)


def make_class_weights(train_ds, num_labels):
    y = collect_flat_labels(train_ds)
    classes = np.arange(num_labels)
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(w, dtype=torch.float32)


class WeightedTrainer(Trainer):
    """Cross-entropy with class weights (fixes all-O collapse from imbalance).
    Trainer is not an nn.Module, so we cannot use register_buffer — store weights as attributes.
    """

    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_weight = class_weights.detach().clone()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self._ce_weight.to(device=logits.device, dtype=logits.dtype)
        loss_fct = nn.CrossEntropyLoss(weight=w, ignore_index=-100)
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
ds = load_english_mu_shroom()
print(ds)

tmp = ds["validation"].train_test_split(test_size=0.2, seed=SEED)
train_split = tmp["train"]
eval_split = tmp["test"]  # held-out 20% of labeled validation (no overlap with train)

text_col, spans_col = "model_output_text", "hard_labels"


def preprocess(batch):
    enc = tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )
    enc["labels"] = [
        spans_to_token_bio(off, sp) for off, sp in zip(enc["offset_mapping"], batch[spans_col])
    ]
    enc.pop("offset_mapping")
    return enc


train_tok = train_split.map(preprocess, batched=True, remove_columns=train_split.column_names)
eval_tok = eval_split.map(preprocess, batched=True, remove_columns=eval_split.column_names)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorForTokenClassification(tokenizer)


def compute_metrics(p):
    """Seqeval (entity) + token F1 for B-HALL and I-HALL + mean 'Hall' F1 for checkpoint selection."""
    preds, labels = p
    preds = np.argmax(preds, axis=-1)

    true_predictions, true_labels = [], []
    flat_pred, flat_true = [], []
    for pred_row, lab_row in zip(preds, labels):
        cp, cl = [], []
        for pr, lb in zip(pred_row, lab_row):
            if lb == -100:
                continue
            cp.append(id2label[int(pr)])
            cl.append(id2label[int(lb)])
            flat_pred.append(int(pr))
            flat_true.append(int(lb))
        true_predictions.append(cp)
        true_labels.append(cl)

    flat_pred = np.array(flat_pred)
    flat_true = np.array(flat_true)

    # Token-level F1 (handles imbalance better than accuracy)
    f1_macro = float(f1_score(flat_true, flat_pred, average="macro", zero_division=0))
    f1_w = float(f1_score(flat_true, flat_pred, average="weighted", zero_division=0))
    f1_o = float(f1_score(flat_true, flat_pred, labels=[0], average="macro", zero_division=0))
    f1_b = float(f1_score(flat_true, flat_pred, labels=[1], average="macro", zero_division=0))
    f1_i = float(f1_score(flat_true, flat_pred, labels=[2], average="macro", zero_division=0))
    f1_hall_mean = (f1_b + f1_i) / 2.0

    out = {
        "f1_macro": f1_macro,
        "f1_weighted": f1_w,
        "f1_O": f1_o,
        "f1_B-HALL": f1_b,
        "f1_I-HALL": f1_i,
        "f1_hall_mean": f1_hall_mean,
    }

    try:
        se = seqeval.compute(
            predictions=true_predictions,
            references=true_labels,
            zero_division=0,
        )
    except TypeError:
        se = seqeval.compute(
            predictions=true_predictions,
            references=true_labels,
        )
    out.update(se)
    return out


class_weights = make_class_weights(train_tok, len(label_list))
print("Class weights (O, B-HALL, I-HALL):", class_weights.tolist())

use_fp16 = torch.cuda.is_available()
args = make_training_args(
    output_dir=OUTPUT_DIR,
    train_bs=4,
    eval_bs=4,
    epochs=NUM_EPOCHS,
    lr=2e-5,
    wd=0.01,
    seed=SEED,
    use_fp16=use_fp16,
)

trainer_kwargs = dict(
    class_weights=class_weights,
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
try:
    trainer = WeightedTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = WeightedTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer.train()

best_dir = os.path.join(OUTPUT_DIR, "best")
os.makedirs(best_dir, exist_ok=True)
trainer.model.save_pretrained(best_dir, safe_serialization=True)
tokenizer.save_pretrained(best_dir)
print("Saved to:", best_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/xlmr-mu-shroom-en/best


## Where are my saved files?

Training writes to the **Colab VM’s local disk**, not to your Google Drive unless you copy them.

- **Path on the VM:** `/content/xlmr-mu-shroom-en/best/` (weights, tokenizer, `config.json`).
- **See them in the UI:** click the **folder icon** in the left sidebar → open **`content`** → **`xlmr-mu-shroom-en`** → **`best`**.
- If the folder is empty, expand **`xlmr-mu-shroom-en`** (not only `best`): checkpoints also live under `OUTPUT_DIR` next to `best`.
- **Important:** `/content/` is **deleted when the runtime disconnects**. Download or copy to Drive before closing the session.

Run the next cell to list files and sizes. Use it to confirm the save worked.

In [3]:
import os

out = "/content/xlmr-mu-shroom-en"
best = os.path.join(out, "best")

def hum(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024.0 or u == "TB":
            return f"{n:.2f} {u}"
        n /= 1024.0

print("Exists:", os.path.isdir(best), "→", best)
if os.path.isdir(out):
    print("\nTop-level under", out, ":")
    for name in sorted(os.listdir(out)):
        p = os.path.join(out, name)
        print(" ", name, "/" if os.path.isdir(p) else "", flush=True)
if os.path.isdir(best):
    print("\nFiles in best/:")
    tot = 0
    for fn in sorted(os.listdir(best)):
        p = os.path.join(best, fn)
        if os.path.isfile(p):
            s = os.path.getsize(p)
            tot += s
            print(f"  {fn:40s}  {hum(s)}")
        else:
            print(f"  {fn}/  (dir)")
    print(f"\nSum of files in best/: ~{hum(tot)}")
else:
    print("best/ not found — training save may have failed or path differs.")

# Optional: download the whole folder as a zip (large ~2GB+; may take minutes)
# SKIP = False
# if not SKIP and os.path.isdir(best):
#     import shutil
#     from google.colab import files
#     zip_path = shutil.make_archive("/content/xlmr-mu-shroom-en-best", "zip", best)
#     files.download(zip_path)

Exists: True → /content/xlmr-mu-shroom-en/best

Top-level under /content/xlmr-mu-shroom-en :
  best /
  checkpoint-10 /
  checkpoint-20 /
  checkpoint-30 /
  checkpoint-40 /
  checkpoint-50 /
  checkpoint-60 /
  checkpoint-70 /
  checkpoint-80 /

Files in best/:
  config.json                               898.00 B
  model.safetensors                         2.08 GB
  tokenizer.json                            16.00 MB
  tokenizer_config.json                     314.00 B

Sum of files in best/: ~2.10 GB


## Inference: spans on new text

In [4]:
import os
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

OUTPUT_DIR = "/content/xlmr-mu-shroom-en"
best_dir = os.path.join(OUTPUT_DIR, "best")
tok = AutoTokenizer.from_pretrained(best_dir, use_fast=True)
mdl = AutoModelForTokenClassification.from_pretrained(
    best_dir,
    ignore_mismatched_sizes=True,
)
mdl.eval()
device = next(mdl.parameters()).device
id2l = mdl.config.id2label

text = "Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China."
enc = tok(text, return_tensors="pt", return_offsets_mapping=True, truncation=True, max_length=256)
offsets = enc.pop("offset_mapping")[0].tolist()
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = mdl(**enc).logits[0]
pred_ids = logits.argmax(-1).tolist()

# Quick sanity: count predicted labels on this sentence
from collections import Counter
cnt = Counter(id2l[int(i)] for i in pred_ids)
print("Predicted label counts:", dict(cnt))

spans = []
cur_s = cur_e = None
for (s, e), pid in zip(offsets, pred_ids):
    if s == e == 0:
        continue
    lab = id2l[int(pid)]
    if lab == "B-HALL":
        if cur_s is not None:
            spans.append([cur_s, cur_e])
        cur_s, cur_e = s, e
    elif lab == "I-HALL" and cur_s is not None:
        cur_e = e
    else:
        if cur_s is not None:
            spans.append([cur_s, cur_e])
            cur_s = cur_e = None
if cur_s is not None:
    spans.append([cur_s, cur_e])

print("Predicted hallucination spans [start, end):", spans)
for a, b in spans:
    print(repr(text[a:b]))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Predicted label counts: {'O': 12, 'B-HALL': 8, 'I-HALL': 1}
Predicted hallucination spans [start, end): [[0, 5], [10, 18], [25, 31], [45, 49], [50, 56], [69, 76], [78, 83]]
'Petra'
'Stoveren'
'silver'
'2008'
'Summer'
'Beijing'
'China'
